In [ ]:
import pandas as pd

In [ ]:
BASE_MODELS = [
    "RandomForestClassifier",
    "SVC",
    "LogisticRegression",
    "DecisionTreeClassifier"
]

DATASETS = ["electricity", 
            "rialto",
            "powersupply", 
            "airlines"
            ]

In [ ]:
df = pd.concat(
    [pd.read_csv(f"../time_elapsed/basemodel: {model}  - dataset: {ds} - with_drift_metrics").assign(dataset=ds)
        for model in BASE_MODELS for ds in DATASETS
     ]
)

df_scoring = pd.concat([pd.read_csv(f"../time_elapsed_scoring/basemodel: {model}  - dataset: {ds} - with_drift_metrics").assign(dataset=ds)
        for model in BASE_MODELS for ds in DATASETS
     ])
df = pd.concat([df,df_scoring],axis=1)
df.drop(columns=["Unnamed: 0"],inplace=True)
print(df.head())

In [ ]:
COLORS = [
    "#e6194b", # red
    "#3cb44b", # green
    "#ffe119", # yellow
    "#4363d8", # blue
    "#f58231", # orange
    "#911eb4", # purple
    "#46f0f0", # cyan
    "#f032e6", # magenta
    "#bcf60c", # lime
    "#fabebe", # pink
    "#008080", # teal
    "#e6beff", # lavender
    "#9a6324", # brown
    "#808080", # grey
]
grouped_df = df.groupby("dataset")
grouped_df = grouped_df.mean()

grouped_df = grouped_df.reindex(sorted(grouped_df.columns), axis=1)
grouped_df = grouped_df.div(grouped_df.sum(axis=1), axis=0)
for dataset, row in grouped_df.iterrows():
    print(f"Dataset: {dataset}")
    print(row.sort_values(ascending=False))
    print("-" * 30)
grouped_df.plot.barh(figsize=(7, 5), color=COLORS,fontsize=16, stacked=True).legend(loc='best', prop={'size': 16}, bbox_to_anchor=(1, -0.1))

In [ ]:
compare = pd.DataFrame()
compare["total"] =  grouped_df.sum(1)
compare["without_drift"] = grouped_df["DBSCANMfesExtractor"]+grouped_df["KmeansMfesExtractor"]+grouped_df["StatsMFesExtractor"]
compare["with_drift"] =compare["total"]-compare["without_drift"]
compare

In [ ]:

compare.plot.barh(figsize=(7,7), fontsize=16, color=COLORS).legend(loc='best', prop={'size': 16})

In [ ]:
x = grouped_df.columns
y = grouped_df.mean()
porcent = 100.*y/y.sum()
labels = ['{0} - {1:1.1f} %'.format(i,j) for i,j in zip(x, porcent)]
item_order = porcent.sort_values(ascending=False).index
print(item_order)
porcent= porcent[item_order]
porcent.plot.barh(figsize=(7, 5), color=COLORS,fontsize=16, stacked=True).legend(loc='best', prop={'size': 16}, bbox_to_anchor=(1, -0.1))

In [ ]:
%store -r all_imps
all_imps = all_imps[["mfe_type", "importance"]].groupby("mfe_type").sum()

In [ ]:
print(all_imps)
all_imps["importance"]= all_imps["importance"]*100

In [ ]:

DRIFT_METRICS_DICT = {
    "KSWINDetector": "KSWIN",
    "JensenShanonDetector": "Jensen-Shannon",
    'StatsMFesExtractor': 'StatsMetrics',
    'BhattacharyyaDetector': "Bhattacharyya",
    "HellingerDistanceDetector": "Hellinger",
    "EMDDetector": "EMD",
    "EnergyDistanceDetector": "Energy"
}

porcent["ClusteringMetrics"] = porcent["KmeansMfesExtractor"]+porcent["DBSCANMfesExtractor"]
porcent.rename(DRIFT_METRICS_DICT,inplace=True)
porcent

In [ ]:
df_porcent = porcent.to_frame().T
print(df_porcent)

In [ ]:
all_imps = all_imps.T
all_imps.rename(columns={'Jensen-Shannon ':'Jensen-Shannon'},inplace=True)
print(all_imps.columns)

In [ ]:
res = pd.concat([df_porcent, all_imps], axis=0)
res
print(res.columns)

In [ ]:
res.drop(columns=["KmeansMfesExtractor","DBSCANMfesExtractor"],inplace=True)

In [ ]:
res

In [ ]:
res = res.T

In [ ]:
res.rename(columns={0: "time"},inplace=True)
res.sort_values(by=["importance"],ascending=False,inplace=True)

In [ ]:

ax = res.plot(
    kind='barh', 
    figsize=(10, 6), 
    color=['#B0C4DE', '#F5E1C8'],
    width=0.8
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.1), ncol=2, frameon=False)

for p in ax.patches:
    ax.annotate(f"{p.get_width():.2f}%", 
                (p.get_width() + 0.5, p.get_y() + 0.1), 
                fontsize=9)
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

In [ ]:
res